# Importing Required Libraries

In [39]:
import pandas as pd

DATA_PATH = r'D:\Data_Analyst_Project\Online_Retail_Business_Analysis\01_Data\Raw_Data'
EXPORT_PATH = r'D:\Data_Analyst_Project\Online_Retail_Business_Analysis\01_Data\Processed_Data'

# Importing Raw Data

In [40]:
order_item_refunds = pd.read_csv(rf'{DATA_PATH}\order_item_refunds.csv')
order_items = pd.read_csv(rf'{DATA_PATH}\order_items.csv')
orders = pd.read_csv(rf'{DATA_PATH}\orders.csv')
products = pd.read_csv(rf'{DATA_PATH}\products.csv')
website_pageviews = pd.read_csv(rf'{DATA_PATH}\website_pageviews.csv')
website_sessions = pd.read_csv(rf'{DATA_PATH}\website_sessions.csv')

In [41]:
orders_clean = orders.copy()
order_items_clean = order_items.copy()
order_item_refunds_clean = order_item_refunds.copy()
products_clean = products.copy()
website_pageviews_clean = website_pageviews.copy()
website_sessions_clean = website_sessions.copy()

# Transforming Dates

In [42]:
orders_clean['created_at'] = pd.to_datetime(orders_clean['created_at'])
order_items_clean['created_at'] = pd.to_datetime(order_items_clean['created_at'])
order_item_refunds_clean['created_at'] = pd.to_datetime(order_item_refunds_clean['created_at'])
products_clean['created_at'] = pd.to_datetime(products_clean['created_at'])
website_pageviews_clean['created_at'] = pd.to_datetime(website_pageviews_clean['created_at'])
website_sessions_clean['created_at'] = pd.to_datetime(website_sessions_clean['created_at'])

In [43]:
print(orders_clean['created_at'].dtype)
print(order_items_clean['created_at'].dtype)
print(order_item_refunds_clean['created_at'].dtype)
print(products_clean['created_at'].dtype)
print(website_pageviews_clean['created_at'].dtype)
print(website_sessions_clean['created_at'].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]


# Data Quality

In [44]:
website_sessions_clean[website_sessions_clean.isnull().any(axis=1)].head()

,website_session_id,created_at,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer
904,905,2012-03-25 07:24:18,905,0,NaN,NaN,NaN,mobile,NaN
914,915,2012-03-25 10:45:03,915,0,NaN,NaN,NaN,mobile,NaN
933,934,2012-03-25 14:23:48,934,0,NaN,NaN,NaN,desktop,NaN
1044,1045,2012-03-26 11:54:49,846,1,NaN,NaN,NaN,desktop,NaN
1050,1051,2012-03-26 12:48:41,152,1,NaN,NaN,NaN,desktop,NaN


In [45]:
website_sessions_clean[['utm_source', 'utm_campaign', 'utm_content', 'http_referer']].isnull().sum()

utm_source      83328
utm_campaign    83328
utm_content     83328
http_referer    39917
dtype: int64

In [46]:
website_sessions_clean[
    website_sessions_clean['utm_source'].isnull()
][['utm_source', 'utm_campaign', 'utm_content', 'http_referer']].head(10)

,utm_source,utm_campaign,utm_content,http_referer
904,NaN,NaN,NaN,NaN
914,NaN,NaN,NaN,NaN
933,NaN,NaN,NaN,NaN
1044,NaN,NaN,NaN,NaN
1050,NaN,NaN,NaN,NaN
1123,NaN,NaN,NaN,https://www.gsearch.com
1146,NaN,NaN,NaN,NaN
1174,NaN,NaN,NaN,https://www.gsearch.com
1247,NaN,NaN,NaN,https://www.gsearch.com
1279,NaN,NaN,NaN,https://www.gsearch.com


In [47]:
website_sessions_clean[
    website_sessions_clean['utm_source'].isnull()
][['utm_source', 'utm_campaign', 'utm_content']].isnull().all(axis=1).value_counts()

True    83328
Name: count, dtype: int64

In [48]:
website_sessions_clean[
    website_sessions_clean['utm_source'].isnull()
]['http_referer'].notnull().value_counts()

http_referer
True     43411
False    39917
Name: count, dtype: int64

In [49]:
website_sessions_clean['utm_source'].value_counts()

utm_source
gsearch       316035
bsearch        62823
socialbook     10685
Name: count, dtype: int64

In [50]:
sessions_with_both = website_sessions_clean[
    website_sessions_clean['utm_source'].notna() &
    website_sessions_clean['http_referer'].notna()
]

In [51]:
sessions_with_both[['utm_source', 'http_referer']].drop_duplicates()

,utm_source,http_referer
0,gsearch,https://www.gsearch.com
1305,bsearch,https://www.bsearch.com
179761,socialbook,https://www.socialbook.com


In [52]:
website_sessions_clean['http_referer'].str.split('.').str[1].head(10)

0    gsearch
1    gsearch
2    gsearch
3    gsearch
4    gsearch
5    gsearch
6    gsearch
7    gsearch
8    gsearch
9    gsearch
Name: http_referer, dtype: object

In [53]:
website_sessions_clean['referer_source'] = website_sessions_clean['http_referer'].str.split('.').str[1]
website_sessions_clean[['http_referer', 'referer_source']].head(10)

,http_referer,referer_source
0,https://www.gsearch.com,gsearch
1,https://www.gsearch.com,gsearch
2,https://www.gsearch.com,gsearch
3,https://www.gsearch.com,gsearch
4,https://www.gsearch.com,gsearch
5,https://www.gsearch.com,gsearch
6,https://www.gsearch.com,gsearch
7,https://www.gsearch.com,gsearch
8,https://www.gsearch.com,gsearch
9,https://www.gsearch.com,gsearch


In [54]:
website_sessions_clean['referer_source'].notna().sum()

np.int64(432954)

In [55]:
(website_sessions_clean['utm_source'] == website_sessions_clean['referer_source']).sum()

np.int64(389543)

In [56]:
website_sessions_clean['utm_source'].notna().sum()

np.int64(389543)

In [57]:
website_sessions_clean[
    website_sessions_clean['utm_source'].isna() &
    website_sessions_clean['referer_source'].notna()
].shape[0]

43411

In [58]:
website_sessions_clean['utm_source'].fillna(website_sessions_clean['referer_source'], inplace=True)

C:\Users\thakr\AppData\Local\Temp\ipykernel_28140\703241178.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  website_sessions_clean['utm_source'].fillna(website_sessions_clean['referer_source'], inplace=True)


In [59]:
website_sessions_clean['utm_source'].isna().sum()

np.int64(39917)

### UTM Source Recovery

UTM source was missing for 83,328 sessions. Investigation showed that:
- All three UTM fields were missing together.
- 43,411 of these sessions had a non-null HTTP referrer.
- The relationship between `utm_source` and the extracted `referer_source` was consistent across all sessions where both were available.

Therefore, missing `utm_source` values were filled using the validated `referer_source`. The remaining 39,917 sessions were left missing because no referrer information was available.

# Referential Integrity Check

In [60]:
orders_clean['website_session_id'].isin(website_sessions_clean['website_session_id']).value_counts()

website_session_id
True    32313
Name: count, dtype: int64

In [61]:
order_items_clean['order_id'].isin(orders_clean['order_id']).value_counts()

order_id
True    40025
Name: count, dtype: int64

In [62]:
order_items_clean['product_id'].isin(products_clean['product_id']).value_counts()

product_id
True    40025
Name: count, dtype: int64

In [63]:
order_item_refunds_clean['order_id'].isin(orders_clean['order_id']).value_counts()

order_id
True    1731
Name: count, dtype: int64

In [64]:
order_item_refunds_clean['order_item_id'].isin(order_items_clean['order_item_id']).value_counts()

order_item_id
True    1731
Name: count, dtype: int64

## Referential Integrity Summary

All tested relationships passed referential integrity checks:

| Relationship | Records Checked | Valid References | Orphan Records |
|---|---:|---:|---:|
| `orders → website_sessions` | 32,313 | 32,313 | 0 |
| `order_items → orders` | 40,025 | 40,025 | 0 |
| `order_items → products` | 40,025 | 40,025 | 0 |
| `order_item_refunds → order_items` | 1,731 | 1,731 | 0 |

**Conclusion:** No orphaned foreign-key records were identified across the tested relationships.

# Feature Engineering

Gross Profit -- Orders

In [65]:
orders_clean['gross_profit'] = orders_clean['price_usd'] - orders_clean['cogs_usd']

In [66]:
orders_clean['gross_profit'].describe()

count    32313.000000
mean        37.636230
std         11.746152
min         20.500000
25%         30.500000
50%         30.500000
75%         37.500000
max         69.000000
Name: gross_profit, dtype: float64

In [67]:
orders_clean[['price_usd', 'cogs_usd', 'gross_profit']].head()

,price_usd,cogs_usd,gross_profit
0,49.99,19.49,30.5
1,49.99,19.49,30.5
2,49.99,19.49,30.5
3,49.99,19.49,30.5
4,49.99,19.49,30.5


Profit Margin -- Orders

In [68]:
orders_clean['profit_margin'] = (orders_clean['gross_profit'] / orders_clean['price_usd']) * 100

In [69]:
orders_clean['profit_margin'].describe()

count    32313.000000
mean        62.561173
std          2.317615
min         61.012202
25%         61.012202
50%         61.012202
75%         63.765941
max         68.493151
Name: profit_margin, dtype: float64

In [70]:
orders_clean[['price_usd', 'gross_profit', 'profit_margin']].head()

,price_usd,gross_profit,profit_margin
0,49.99,30.5,61.012202
1,49.99,30.5,61.012202
2,49.99,30.5,61.012202
3,49.99,30.5,61.012202
4,49.99,30.5,61.012202


In [71]:
orders_clean['profit_margin'].value_counts()

profit_margin
61.012202    18104
62.510418     3924
63.765941     3142
68.493151     2039
64.596791     2036
61.829424      944
64.458769      680
68.439063      662
68.356119      534
65.106624      248
Name: count, dtype: int64

Gross Profit -- Order Items

In [72]:
order_items_clean['gross_profit'] = order_items_clean['price_usd'] - order_items_clean['cogs_usd']

Profit Margin -- Order Items

In [73]:
order_items_clean['profit_margin'] = (order_items_clean['gross_profit'] / order_items_clean['price_usd']) * 100

Dropping referer_source column from Website Sessions

In [74]:
website_sessions_clean.drop(columns='referer_source', inplace=True)

In [75]:
website_pageviews_clean.head(10)

,website_pageview_id,created_at,website_session_id,pageview_url
0,1,2012-03-19 08:04:16,1,/home
1,2,2012-03-19 08:16:49,2,/home
2,3,2012-03-19 08:26:55,3,/home
3,4,2012-03-19 08:37:33,4,/home
4,5,2012-03-19 09:00:55,5,/home
5,6,2012-03-19 09:05:46,6,/home
6,7,2012-03-19 09:06:27,7,/home
7,8,2012-03-19 09:10:08,6,/products
8,9,2012-03-19 09:10:52,6,/the-original-mr-fuzzy
9,10,2012-03-19 09:14:02,6,/cart


# Exporting Cleaned Data

In [76]:
orders_clean.to_csv(rf'{EXPORT_PATH}\orders_clean.csv', index=False)
order_items_clean.to_csv(rf'{EXPORT_PATH}\order_items_clean.csv', index=False)
order_item_refunds_clean.to_csv(rf'{EXPORT_PATH}\order_item_refunds_clean.csv', index=False)
products_clean.to_csv(rf'{EXPORT_PATH}\products_clean.csv', index=False)
website_pageviews_clean.to_csv(rf'{EXPORT_PATH}\website_pageviews_clean.csv', index=False)
website_sessions_clean.to_csv(rf'{EXPORT_PATH}\website_sessions_clean.csv', index=False)